**Imports**

In [1]:
import pandas as pd
import numpy as np
import torch
import os
import joblib
import time
import warnings
warnings.filterwarnings('ignore')
 
from transformers import AutoTokenizer, AutoModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import matplotlib.pyplot as plt
 
RANDOM_STATE = 42
 
# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU found. Embedding extraction will be slow.")
    print("Consider using Google Colab with T4 GPU for this notebook.")

Using device: cpu
Consider using Google Colab with T4 GPU for this notebook.


**Load splits**

In [2]:
# Adjust path if running in Colab after uploading splits/
SPLITS_PATH = '../data/splits/'
 
X_train = pd.read_csv(SPLITS_PATH + 'X_train.csv')
X_val   = pd.read_csv(SPLITS_PATH + 'X_val.csv')
y_train = pd.read_csv(SPLITS_PATH + 'y_train.csv').squeeze()
y_val   = pd.read_csv(SPLITS_PATH + 'y_val.csv').squeeze()
 
print(f"Train: {len(X_train):,}  |  Val: {len(X_val):,}")
 
# Extract the symptom_text column — this is what goes into the Transformers
texts_train = X_train['symptom_text'].tolist()
texts_val   = X_val['symptom_text'].tolist()
 
print(f"\nSample input text:")
print(texts_train[0])

Train: 41,555  |  Val: 8,868

Sample input text:
Species: Dolphin. Breed: Bottlenose. Age: 15 years. Gender: Male. Weight: 548.4 kg. Primary symptom: appetite loss. Additional symptoms: diarrhea, labored breathing, lameness, nasal discharge, eye discharge. Symptoms present: appetite loss, diarrhea, labored breathing, lameness, nasal discharge, eye discharge. Symptoms absent: vomiting, coughing, skin lesions. Body temperature: 36.8 C. Danger level: Moderate.


**Feature extraction function**

Extract CLS token embeddings from a pre-trained Transformer model.
 
The CLS token ([CLS]) is the first token in every BERT input.
After processing through all 12 Transformer layers, its vector
representation summarises the entire input sequence — making it
ideal as a fixed-length feature vector for classification.
 
    Args:
        texts:      List of symptom text strings
        model_name: HuggingFace model identifier
        batch_size: How many texts to process at once (reduce if OOM)
        max_length: Max tokens per text (256 covers most vet descriptions)
 
    Returns:
        numpy array of shape (n_samples, 768)

In [3]:
def extract_cls_embeddings(texts, model_name, batch_size=32, max_length=256):
    print(f"\nLoading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModel.from_pretrained(model_name)
    model.eval()
    model.to(device)
 
    all_embeddings = []
    n_batches = (len(texts) + batch_size - 1) // batch_size
    start_time = time.time()
 
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
 
        # Tokenize
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
 
        # Forward pass — no gradient needed (we're just extracting, not training)
        with torch.no_grad():
            outputs = model(**encoded)
 
        # CLS token = first token of last hidden state
        cls_vectors = outputs.last_hidden_state[:, 0, :]  # shape: (batch, 768)
        all_embeddings.append(cls_vectors.cpu().numpy())
 
        # Progress
        batch_num = i // batch_size + 1
        if batch_num % 50 == 0 or batch_num == n_batches:
            elapsed = time.time() - start_time
            pct = batch_num / n_batches * 100
            print(f"  Batch {batch_num}/{n_batches} ({pct:.0f}%) — {elapsed:.0f}s elapsed")
 
    embeddings = np.vstack(all_embeddings)
    total_time = time.time() - start_time
    print(f"Done. Shape: {embeddings.shape}  Time: {total_time:.0f}s")
 
    # Free GPU memory
    del model
    torch.cuda.empty_cache() if device.type == 'cuda' else None
 
    return embeddings

**Three Transformer models to compare**

In [4]:
# These are the three models your IR commits to evaluating (Objective 1)
MODELS = {
    'BERT':        'bert-base-uncased',
    'BioBERT':     'dmis-lab/biobert-v1.1',
    'PubMedBERT':  'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract',
}
 
os.makedirs('../models/embeddings', exist_ok=True)

**Extract and save embeddings for each model**

In [ ]:
# Embeddings are saved immediately so you don't lose work if it crashes
 
for model_short_name, model_hf_name in MODELS.items():
    train_path = f'../models/embeddings/{model_short_name}_train.npy'
    val_path   = f'../models/embeddings/{model_short_name}_val.npy'
 
    # Skip if already extracted (allows resuming after crash)
    if os.path.exists(train_path) and os.path.exists(val_path):
        print(f"{model_short_name}: embeddings already exist, skipping extraction.")
        continue
 
    print(f"\n{'='*60}")
    print(f"Extracting {model_short_name} ({model_hf_name})")
    print(f"{'='*60}")
 
    train_emb = extract_cls_embeddings(texts_train, model_hf_name)
    val_emb   = extract_cls_embeddings(texts_val,   model_hf_name)
 
    np.save(train_path, train_emb)
    np.save(val_path,   val_emb)
    print(f"Saved: {train_path}")
    print(f"Saved: {val_path}")


Extracting BERT (bert-base-uncased)

Loading bert-base-uncased...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1467.28it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Quick classifier comparison using a Random Forest**

In [ ]:
def quick_rf_eval(model_name):
    """Train a fast Random Forest and evaluate on validation set."""
    print(f"\n--- {model_name} ---")
    train_emb = np.load(f'../models/embeddings/{model_name}_train.npy')
    val_emb   = np.load(f'../models/embeddings/{model_name}_val.npy')
 
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        class_weight='balanced',   # Handle 8941:1 imbalance
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    rf.fit(train_emb, y_train)
 
    y_pred = rf.predict(val_emb)
    acc       = accuracy_score(y_val, y_pred)
    f1_macro  = f1_score(y_val, y_pred, average='macro',  zero_division=0)
    f1_micro  = f1_score(y_val, y_pred, average='micro',  zero_division=0)
    f1_weight = f1_score(y_val, y_pred, average='weighted', zero_division=0)
 
    print(f"  Accuracy:          {acc:.4f}")
    print(f"  F1 Macro:          {f1_macro:.4f}  ← primary metric")
    print(f"  F1 Micro:          {f1_micro:.4f}")
    print(f"  F1 Weighted:       {f1_weight:.4f}")
 
    return {
        'Model': model_name,
        'Accuracy': round(acc, 4),
        'F1_Macro': round(f1_macro, 4),
        'F1_Micro': round(f1_micro, 4),
        'F1_Weighted': round(f1_weight, 4),
    }
 
 
# Run comparison
comparison_results = []
for name in MODELS.keys():
    result = quick_rf_eval(name)
    comparison_results.append(result)
 
comparison_df = pd.DataFrame(comparison_results).sort_values('F1_Macro', ascending=False)
print("\n" + "="*60)
print("TRANSFORMER COMPARISON RESULTS (Random Forest baseline)")
print("="*60)
print(comparison_df.to_string(index=False))
os.makedirs('../results', exist_ok=True)
comparison_df.to_csv('../results/transformer_comparison.csv', index=False)

**Decision — pick the best Transformer**

In [ ]:
best_transformer = comparison_df.iloc[0]['Model']
best_f1          = comparison_df.iloc[0]['F1_Macro']
print(f"\n✅ BEST TRANSFORMER: {best_transformer} (F1 Macro = {best_f1:.4f})")
print(f"Use '{MODELS[best_transformer]}' as your Transformer in Notebook 03.")
 
# Save the decision
with open('../results/best_transformer.txt', 'w') as f:
    f.write(f"best_transformer={best_transformer}\n")
    f.write(f"best_hf_name={MODELS[best_transformer]}\n")
    f.write(f"f1_macro={best_f1}\n")

**Visualise comparison**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
 
metrics = ['Accuracy', 'F1_Macro', 'F1_Micro', 'F1_Weighted']
colors  = ['#3B8BD4', '#1D9E75', '#EF9F27']
 
# Bar chart: F1 Macro comparison
ax = axes[0]
bars = ax.bar(comparison_df['Model'], comparison_df['F1_Macro'], color=colors)
ax.set_title('F1 Macro Score by Transformer Model')
ax.set_ylabel('F1 Macro Score')
ax.set_ylim([0, 1])
ax.axhline(y=0.80, color='red', linestyle='--', alpha=0.7, label='Target (0.80)')
ax.legend()
for bar, val in zip(bars, comparison_df['F1_Macro']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11)
 
# Table: all metrics
ax2 = axes[1]
ax2.axis('off')
table_data = comparison_df[['Model', 'Accuracy', 'F1_Macro', 'F1_Micro']].values
col_labels = ['Model', 'Accuracy', 'F1 Macro', 'F1 Micro']
table = ax2.table(cellText=table_data, colLabels=col_labels,
                  loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2)
ax2.set_title('Full Comparison Table')
 
plt.tight_layout()
plt.savefig('../results/transformer_comparison_chart.png', dpi=150)
plt.show()
 
print("\nNOTEBOOK 02 COMPLETE — proceed to 03_modeling.py")
print(f"Use embeddings: ../models/embeddings/{best_transformer}_train.npy")